In [3]:
import os
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

In [17]:
# 1. Load Dataset Berita Detik.com

dataset_path = os.path.join("01_crawling", "data_berita_detik.csv")
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset tidak ditemukan di: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Total Dokumen Berita : {len(df)} berita")
print(f"Distribusi Kelas     :\n{df['label'].value_counts().to_string()}\n")

Total Dokumen Berita : 200 berita
Distribusi Kelas     :
label
sport      100
finance    100



In [18]:
# 2. Kumpulan kata tidak baku / slang / singkatan yang umum di berita online
daftar_kata_tidak_baku = {
    "gak", "gk", "ga", "nggak", "ngga", "enggak", "kagak",
    "udah", "udh", "dah",
    "emang", "emg", "mmg",
    "banget", "bgt", "bngt",
    "aja", "aj",
    "gimana", "gmn",
    "gitu", "gtu",
    "kayak", "kyk",
    "biar", "byr",
    "doang", "dong", "dng",
    "nih", "neh",
    "tuh",
    "sih", "si",
    "yg", "yng",
    "dgn", "dg",
    "utk",
    "dlm",
    "tdk", "gk",
    "blm",
    "krn", "krna",
    "bs", "bsa",
    "jg", "jga",
    "sm", "sama2",
    "org",
    "trs", "trus", "terus2an",
    "lg", "lgi",
    "dr",
    "pd",
    "klo", "kalo",
    "ama",
    "bkn",
    "tp", "tpi",
    "dll",
    "dsb",
    "dst",
    "dkk",
    "wkwk", "wkwkwk", "haha", "hihi", "hehe",
    "btw", "fyi", "omg", "lol",
    "nyokap", "bokap",
    "cuy", "cuk", "bro", "sis", "gan",
    "gue", "gw", "gwa", "lo", "lu",
    "ngapain", "napa", "kenapa2",
    "mah", "atuh", "teh",
}

In [19]:
# 3. preprocessing
# Membuang tanda baca dan kata tidak baku (Tanpa Lowercase)

def preprocess_versi_1(teks):
    # Ambil semua kata dan angka, buang tanda baca
    list_token = re.findall(r'\b\w+\b', teks)
    # Buang kata tidak baku
    clean_token = [kata for kata in list_token if kata.lower() not in daftar_kata_tidak_baku]
    return clean_token

In [20]:
# 4. Fungsi Representasi Vektor Dokumen (Mean Pooling)

# Menghitung rata-rata vektor kata dari model Skip-Gram untuk merepresentasikan satu dokumen
def hitung_vektor_dokumen(list_token, model_skipgram, dimensi_vektor):
    vektor_kata_valid = [model_skipgram.wv[kata] for kata in list_token if kata in model_skipgram.wv]
    if len(vektor_kata_valid) > 0:
        return np.mean(vektor_kata_valid, axis=0)
    else:
        return np.zeros(dimensi_vektor)

In [21]:
# 5. Tokenisasi

data_token = [preprocess_versi_1(isi) for isi in df['isi_berita']]

total_kata = sum(len(kalimat) for kalimat in data_token)
kosakata_unik = len(set(kata for kalimat in data_token for kata in kalimat))
print(f"Total Kata Keseluruhan : {total_kata:,} kata")
print(f"Total Kosakata Unik    : {kosakata_unik:,} kata")

Total Kata Keseluruhan : 66,952 kata
Total Kosakata Unik    : 8,932 kata


In [22]:
# 6. Training Model SKIP-GRAM (Jaringan Saraf Tiruan)

dimensi_vektor = 100
ukuran_jendela = 5

print(f"    - Parameter Dimensi Vektor (vector_size) : {dimensi_vektor}")
print(f"    - Parameter Tetangga Konteks (window)   : {ukuran_jendela} (kiri & kanan)")
print(f"    - Parameter Arsitektur                   : sg = 1 (Skip-Gram: Target -> Context)")
print(f"    - Parameter Min Count                   : 1 (semua kata diikutkan)")

model_skipgram = Word2Vec(
    sentences=data_token,
    vector_size=dimensi_vektor,
    window=ukuran_jendela,
    min_count=1,
    sg=1,              # 1 = SKIP-GRAM
    epochs=30,         # Iterasi pelatihan JST
    seed=42,
    workers=1
)

    - Parameter Dimensi Vektor (vector_size) : 100
    - Parameter Tetangga Konteks (window)   : 5 (kiri & kanan)
    - Parameter Arsitektur                   : sg = 1 (Skip-Gram: Target -> Context)
    - Parameter Min Count                   : 1 (semua kata diikutkan)


In [23]:
# 7. Ekstraksi Vektor Dokumen (Mean Pooling)

fitur_X = np.array([hitung_vektor_dokumen(token, model_skipgram, dimensi_vektor) for token in data_token])
label_y = df['label'].values
print(f"    - Matriks Fitur X : {fitur_X.shape} ({len(df)} dokumen x {dimensi_vektor} dimensi fitur)")

    - Matriks Fitur X : (200, 100) (200 dokumen x 100 dimensi fitur)


In [24]:
# 8. Export CSV untuk klasifikasi

nama_kolom = [f"dim_{i+1}" for i in range(dimensi_vektor)]
df_vektor = pd.DataFrame(fitur_X, columns=nama_kolom)
df_vektor['label'] = label_y

output_csv = "vektor_skipgram_versi_1.csv"
df_vektor.to_csv(output_csv, index=False)

print(f"    - Jumlah baris  : {len(df_vektor)}")
print(f"    - Jumlah kolom  : {len(df_vektor.columns)} (100 dimensi vektor + 1 label)")

    - Jumlah baris  : 200
    - Jumlah kolom  : 101 (100 dimensi vektor + 1 label)
